# Preparing the BRFSS Data

## Loading

Downloaded March 2, 2021

https://www.cdc.gov/brfss/annual_data/annual_2019.html

Codebook: https://www.cdc.gov/brfss/annual_data/2019/pdf/codebook19_llcp-v2-508.HTML

In [1]:
import pandas as pd
import numpy as np

In [2]:
url = 'https://www.cdc.gov/brfss/annual_data/2019/llcp_varlayout_19_onecolumn.html'
tables = pd.read_html(url)
column_info = tables[0]
column_info.index = column_info['Variable Name']

In [3]:
column_info['Starting Column'] -= 1
column_info['Ending Column'] = (column_info['Starting Column'] + 
                                column_info['Field Length'])
column_info.head()

,Starting Column,Variable Name,Field Length,Ending Column
Variable Name,,,,
_STATE,0,_STATE,2,2
FMONTH,16,FMONTH,2,18
IDATE,18,IDATE,8,26
IMONTH,18,IMONTH,2,20
IDAY,20,IDAY,2,22


In [4]:
names = ['SEQNO', 'HTM4', 'WTKG3', '_SEX', '_AGEG5YR', 
         '_SMOKER3', '_VEGESU1', '_INCOMG', '_EDUCAG', '_LLCPWT']

names = ['SEQNO', 'HTM4', 'WTKG3', '_SEX', '_AGEG5YR',
         '_VEGESU1', '_INCOMG', '_LLCPWT']

In [5]:
cols = ['Starting Column', 'Ending Column']
colspecs_df = column_info.loc[names, cols]
colspecs_df

,Starting Column,Ending Column
Variable Name,,
SEQNO,35,45
HTM4,1989,1992
WTKG3,1992,1997
_SEX,1979,1980
_AGEG5YR,1980,1982
_VEGESU1,2142,2148
_INCOMG,2005,2006
_LLCPWT,1750,1760


In [6]:
colspecs = list(colspecs_df.itertuples(index=False, name=None))
colspecs

[(35, 45),
 (1989, 1992),
 (1992, 1997),
 (1979, 1980),
 (1980, 1982),
 (2142, 2148),
 (2005, 2006),
 (1750, 1760)]

In [7]:
import gzip

data_file = 'LLCP2019.ASC.gz'
fp = gzip.open(data_file)

%time brfss = pd.read_fwf(fp, names=names, colspecs=colspecs)

CPU times: user 8.7 s, sys: 172 ms, total: 8.87 s
Wall time: 8.87 s


In [8]:
brfss.shape

(418268, 8)

In [9]:
brfss.head()

,SEQNO,HTM4,WTKG3,_SEX,_AGEG5YR,_VEGESU1,_INCOMG,_LLCPWT
0,2019000001,157.0,6985.0,2,13,114.0,2,135.304080
1,2019000002,163.0,4899.0,2,11,121.0,3,1454.882220
2,2019000003,165.0,8618.0,2,10,164.0,5,215.576852
3,2019000004,165.0,5534.0,2,13,NaN,4,261.282838
4,2019000005,152.0,4990.0,2,13,178.0,9,535.270103


To map from age groups to ages, I use the following formula, which maps each code to the approximate midpoint of its range.
See https://www.cdc.gov/brfss/annual_data/2019/pdf/codebook19_llcp-v2-508.HTML

In [10]:
df = pd.DataFrame(index=range(1,14))
df['AGE'] = df.index * 5 + 17
df

,AGE
1,22
2,27
3,32
4,37
5,42
6,47
7,52
8,57
9,62
10,67


In [11]:
brfss['WTKG3'] /= 100
brfss['_AGEG5YR'] = brfss['_AGEG5YR'].replace(14, np.nan)
brfss['AGE'] = brfss['_AGEG5YR'] * 5 + 17

brfss['AGE'].value_counts().sort_index()

AGE
22.0    25098
27.0    20817
32.0    23058
37.0    24724
42.0    24258
47.0    26075
52.0    31768
57.0    38902
62.0    44456
67.0    45206
72.0    41535
77.0    29767
82.0    35916
Name: count, dtype: int64

In [12]:
height = brfss['HTM4']
bins = np.arange(0, height.max(), 10)
brfss['_HTM4G10'] = pd.cut(brfss['HTM4'], bins=bins, labels=bins[:-1]).astype(float)

In [13]:
!rm brfss.hdf

rm: cannot remove 'brfss.hdf': No such file or directory


In [14]:
brfss.to_hdf('brfss.hdf', key='brfss', complevel=6)

In [15]:
%time brfss = pd.read_hdf('brfss.hdf', 'brfss')

CPU times: user 184 ms, sys: 8 ms, total: 192 ms
Wall time: 192 ms


In [16]:
!ls -lh brfss.hdf

-rw-rw-r-- 1 downey downey 10M Sep 24 11:00 brfss.hdf


In [17]:
!cp brfss.hdf ../data

# add the new file to the repo from the command line

*Elements of Data Science*

Copyright 2021 [Allen B. Downey](https://allendowney.com)

License: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)